# Fyre opp Neo4j
Sjekke at vi kan kjøre Neo4J

Starte med
```
sudo apt install podman
```

In [164]:
%%bash
PWD=$(pwd)
mkdir -p neo4j
podman run \
    -p 7474:7474 -p 7687:7687 \
    --userns=keep-id \
    -e NEO4J_dbms_security_procedures_unrestricted=apoc.* \
    -e NEO4J_dbms_security_procedures_allowlist=apoc.* \
    -e NEO4J_apoc_import_file_enabled=true \
    -v $PWD/neo4j/data:/data:Z \
    -v $PWD/neo4j/logs:/logs:Z \
    -v $PWD/neo4j/import:/import:Z \
    -v $PWD/neo4j/plugins:/plugins:Z \
    -e NEO4J_AUTH=neo4j/password \
    -e NEO4J_PLUGINS='["apoc", "gds"]' \
    -d docker.io/library/neo4j:latest

f0d21d6cddc672fd9d7b0d699c43779848ed74294fe0a6c21563ccd82443ee4d


Når man er ferdig er det bare å kopiere identifikatoren over inn i neste kall

In [163]:
%%bash
podman kill d1349c6d65b5e1ac85ac00826fbe43f4e4e51a6553dd897b456c304a2d69734d

d1349c6d65b5e1ac85ac00826fbe43f4e4e51a6553dd897b456c304a2d69734d


Gi databasen litt tid til å starte

In [165]:
from neo4j import GraphDatabase

# 1. Define connection details
URI = "bolt://localhost:7687"
AUTH = ("neo4j", "password")

driver = GraphDatabase.driver(URI, auth=AUTH)
driver.verify_connectivity()
print("Connection Successful!")

Connection Successful!


Sjekke at vi har APOC tilgjengelig

In [166]:
records, summary, keys = driver.execute_query(
    """RETURN apoc.version() AS version;""")

print(f"Server Address: {summary.server.address}")
print("Keys:")
for k in range(len(keys)):
    print(f"\t{keys[k]}: {records[k]}")
#

print("Grafen")
print(f"\tNye noder: {summary.counters.nodes_created}")
print(f"\tNye kanter: {summary.counters.relationships_created}")

print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
print(f"\tÅ konsumere: {summary.result_consumed_after}ms")



Server Address: 127.0.0.1:7687
Keys:
	version: <Record version='2025.11.2'>
Grafen
	Nye noder: 0
	Nye kanter: 0
Ressursbruk
	Kjøringen: 158ms
	Å konsumere: 3ms


Ofte vil vi lage ting i Networkx og så laste det inn slik at andre kan bruke det.

In [167]:
import gzip
import networkx as nx

with gzip.open("data/email.edgelist.txt.gz", "rt") as fd:
    G = nx.read_edgelist(fd, create_using=nx.DiGraph())
#
G.remove_edges_from(nx.selfloop_edges(G))
print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")

nx.set_node_attributes(G, ":Person", name="labels")
nx.set_edge_attributes(G, "EPOST", name="label")

# Skriv ut grafen
nx.write_graphml(G, "neo4j/import/large_graph.graphml", named_key_ids=True)


Nodes: 57194
Edges: 103083


Les inn

In [168]:
records, summary, keys = driver.execute_query(
    """CALL apoc.import.graphml("large_graph.graphml", {storeNodeIds: true, readLabels: true})""")
for record in records:
    # Converts the entire record into a Python dict
    record_dict = record.data()
#
for k in keys:
    print(f"\t{k}: {record_dict[k]}")


print("Grafen")
print(f"\tNye noder: {summary.counters.nodes_created}")
print(f"\tNye kanter: {summary.counters.relationships_created}")

print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
print(f"\tÅ konsumere: {summary.result_consumed_after}ms")

	file: large_graph.graphml
	source: file
	format: graphml
	nodes: 57194
	relationships: 103083
	properties: 0
	time: 2535
	rows: 0
	batchSize: -1
	batches: 0
	done: True
	data: None
Grafen
	Nye noder: 0
	Nye kanter: 0
Ressursbruk
	Kjøringen: 163ms
	Å konsumere: 2549ms


Sjekke en velkjent node:

In [169]:
records, summary, keys = driver.execute_query(
    """MATCH p=()-->(:Person {id:"6"}) RETURN p;
""")
# Returnerer et sett av STIER som ender i noden identifisert med id=6
for r in records:
    innhold = r.data()
    print(innhold)
    break
#
records, summary, keys = driver.execute_query(
    """MATCH (p)-->(:Person {id:"6"}) RETURN p;
""")
# Returnerer et sett av NODER som har relasjoner til noden identifisert med id=6
for r in records:
    innhold = r.data()
    print(innhold)
    break
#
    

{'p': [{'id': '24805'}, 'EPOST', {'id': '6'}]}
{'p': {'id': '24805'}}


For å lete etter klikker skal vi først merke noder vi ikke er interessert i

In [172]:
# Legg inn
records, summary, keys = driver.execute_query(
    """
MATCH (n:Person)
SET n.antallKanter = COUNT { (n)--() }
""")
for r in records:
    innhold = r.data()
    print(innhold)
    break
#
records, summary, keys = driver.execute_query(
    """
    CREATE INDEX
    IF NOT EXISTS 
    FOR (n:Person) ON (n.antallKanter);
""")
for r in records:
    innhold = r.data()
    print(innhold)
    break
#


Så leter vi etter den største klikken (som vi vet skal ha 6 elementer)

In [173]:
records, summary, keys = driver.execute_query(
    """
MATCH (n:Person)
WHERE n.antallKanter >= 9
WITH collect(n) AS nodes
CALL apoc.nodes.cliques(nodes) YIELD clique
RETURN clique""")
for r in records:
    innhold = r.data()
    print(innhold)
    break
#MATCH (n:Person)


ClientError: {neo4j_code: Neo.ClientError.Procedure.ProcedureNotFound} {message: There is no procedure with the name `apoc.nodes.cliques` registered for this database instance. Please ensure you've spelled the procedure name correctly and that the procedure is properly deployed.} {gql_status: 42001} {gql_status_description: error: syntax error or access rule violation - invalid syntax}